# Public Signals Overlay Intelligence (Reference Basket)

    Takes a reference equity basket (e.g. mega-cap or index constituents) and overlays multiple public signals: earnings/revision events, congressional + 13F flow changes, relevant macro releases, COT positioning in related futures, and vol regime context.

    Produces a simple prioritized 'what to watch' view — the kind of daily mosaic a pod or family office desk would consume.

    **Category:** Multi-source daily intelligence / decision support

    **Primary API calls used:**
    - Equity calendar + estimates (earnings/revision)
    - Congress + 13F / regulatory ownership
    - Macro calendar and key series
    - COT / positioning
    - Pricing + VIX for context

    No user PMS, no IBOR, no private holdings. Purely public data fusion.

In [ ]:
import os
from typing import Any
from datetime import datetime, timedelta
import pandas as pd
from quantjourney.sdk import QuantJourneyAPI
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2024-01-01')
END = os.getenv('QJ_EXAMPLE_END', '2026-06-06')

def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(p):
    v = unwrap(p)
    if isinstance(v, list):
        return v
    if isinstance(v, dict):
        for k in ('rows', 'data', 'items', 'earnings', 'filings', 'trades'):
            if isinstance(v.get(k), list):
                return v[k]
        return [v]
    return []
basket = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'JPM']
print('=== Public Signals Overlay Intelligence ===\nBasket:', basket)
earnings = qj.fmp.get_earnings_calendar(from_date=(datetime.now() - timedelta(days=45)).date(), to_date=datetime.now().date())
earn_df = pd.DataFrame(as_rows(earnings))
if not earn_df.empty:
    earn_df = earn_df[earn_df.get('symbol', '').isin(basket)]
print('Recent earnings in basket:', len(earn_df))
congress_hits = []
for sym in basket[:4]:
    for src in [qj.fmp.get_house_trades, qj.fmp.get_senate_trades]:
        t = as_rows(src(symbol=sym))
        if t:
            congress_hits.append({'symbol': sym, 'count_recent': len(t)})
print('Congress activity sample:', pd.DataFrame(congress_hits).head())
macro_events = getattr(qj, 'macro', qj).get_economic_events(from_date=str(datetime.now().date() - timedelta(days=10)))
print('Recent macro events (count):', len(as_rows(macro_events)))
vix = qj.cboe.get_vix_data()
if vix is not None:
    print('VIX data available for regime overlay.')
print('\nThis notebook produces the raw signals a human analyst would combine into a one-pager.\nIn a fuller version: rank by impact, attach request_ids, and output a clean markdown/HTML brief.')


## Notes

Candidate for daily intelligence style output using only public multi-source signals (earnings/estimates, congress, 13F, macro calendar, COT, vol).
Feed a real holdings list (from 13F example or index constituents) to turn it into a book-relevant packet.
No PMS or IBOR data is accessed or required.